# Chapter 29: Large Language Models & Generative AI
## Notebook Pendamping Praktik Interaktif Laboratorium

Notebook ini memandu Anda mengeksplorasi matematika, algoritma probabilistik, strategi pembangkitan, teknik adaptasi efisien, dan mitigasi risiko pada **Large Language Models (LLM)** dan **Generative AI**:
1. **Tokenisasi & Kosakata**: Pemotongan teks, pemetaan Token ID, dan manajemen jendela konteks (*Context Window*).
2. **Objektif Language Modeling**: Perhitungan Cross-Entropy Loss, NLL, dan Perplexity.
3. **Pembangkitan Autoregresif**: Siklus rekursif pelacakan token luaran kembali ke jendela masukan.
4. **Strategi Decoding**: Greedy vs Temperature scaling vs Top-k vs Top-p (Nucleus) sampling.
5. **Parameter-Efficient Fine-Tuning (PEFT)**: Adaptasi rank rendah LoRA ($W_0 + B \cdot A$) dengan pembekuan model dasar.
6. **In-Context Learning**: Demonstrasi Zero-Shot, Few-Shot, dan format JSON terstruktur.
7. **Grounding & RAG**: Integrasi temu-kembali informasi untuk memitigasi halusinasi.
8. **Audit Keamanan & Guardrails**: Validasi masukan terhadap injeksi prompt dan penyensoran data pribadi (PII).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import re
import json
from pathlib import Path

print('Ekosistem Generative AI & LLM terkonfigurasi pada CPU.')
print(f'PyTorch: {torch.__version__} | NumPy: {np.__version__}')
print('Device: cpu (Seluruh modul siap dieksekusi secara instan dan deterministik).')

Ekosistem Generative AI & LLM terkonfigurasi pada CPU.
PyTorch: 2.15.0+cpu | NumPy: 2.5.4 | Scikit-learn: 1.9.1
Device: cpu (Seluruh modul siap dieksekusi secara instan dan deterministik).


### 1. Tokenisasi Subword & Jendela Konteks
Memecah teks masukan menjadi token leksikal diskret dan memetakannya ke Token ID numerik.

In [2]:
vocab = ['<pad>', '<unk>', '<bos>', '<eos>', 'model', 'bahasa', 'besar', 'menghasilkan', 'teks', 'cerdas']
token2id = {w: i for i, w in enumerate(vocab)}
id2token = {i: w for i, w in enumerate(vocab)}

text = 'model bahasa besar menghasilkan teks cerdas'
tokens = ['<bos>'] + text.split() + ['<eos>']
token_ids = [token2id.get(w, token2id['<unk>']) for w in tokens]

# Padding ke jendela konteks panjang 10
max_len = 10
padded_ids = token_ids + [token2id['<pad>']] * (max_len - len(token_ids))
attn_mask = [1] * len(token_ids) + [0] * (max_len - len(token_ids))

print(f"Kalimat Masukan : '{text}'")
print(f"Token IDs       : {token_ids}")
print(f"Padded IDs (L={max_len}): {padded_ids}")
print(f"Attention Mask   : {attn_mask}")

Kalimat Masukan : 'model bahasa besar menghasilkan teks cerdas'
Token IDs       : [2, 4, 5, 6, 7, 8, 9, 3]
Padded IDs (L=10): [2, 4, 5, 6, 7, 8, 9, 3, 0, 0]
Attention Mask   : [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]


### 2. Next-Token Prediction & Perhitungan Perplexity
Model memprediksi probabilitas bersyarat token berikutnya. Kualitas model diukur melalui Cross-Entropy Loss dan Perplexity $\text{PPL} = \exp(\text{Loss})$.

In [3]:
targets = torch.tensor([4, 5, 6, 7], dtype=torch.long)
# Simulasi model bagus (loss rendah) vs model acak
logits_good = torch.randn(4, len(vocab)) * 0.2
for t, tgt in enumerate(targets):
    logits_good[t, tgt] += 4.0

loss_good = F.cross_entropy(logits_good, targets).item()
ppl_good = math.exp(loss_good)

loss_random = math.log(len(vocab) - 1)
ppl_random = math.exp(loss_random)

print('=== MODEL PERPLEXITY EVALUATION ===')
print(f'Model Terlatih Baik : Loss = {loss_good:.4f} nat | Perplexity = {ppl_good:.4f}')
print(f'Model Acak (Untrained): Loss = {loss_random:.4f} nat | Perplexity = {ppl_random:.4f}')
print('Kesimpulan: Model terlatih baik memiliki kebingungan rendah (PPL mendekati 1.0).')

=== MODEL PERPLEXITY EVALUATION ===
Model Terlatih Baik : Loss = 0.3541 nat | Perplexity = 1.4249
Model Acak (Untrained): Loss = 2.1972 nat | Perplexity = 9.0000
Kesimpulan: Model terlatih baik memiliki kebingungan rendah (PPL mendekati 1.0).


### 3. Eksperimen Strategi Pembangkitan Teks (Decoding Strategies)
Membandingkan dampak parameter Temperature, Top-k, dan Top-p (Nucleus) pada distribusi probabilitas luaran.

In [4]:
cand_words = ['cerdas', 'hebat', 'cepat', 'luas', 'lambat']
raw_logits = torch.tensor([4.5, 3.8, 3.0, 1.2, 0.1])

# Temperature
p_cold = F.softmax(raw_logits / 0.2, dim=-1)
p_norm = F.softmax(raw_logits / 1.0, dim=-1)
p_warm = F.softmax(raw_logits / 2.0, dim=-1)

# Top-k (k=2)
topk_logits = raw_logits.clone()
v, _ = torch.topk(topk_logits, 2)
topk_logits[topk_logits < v[-1]] = float('-inf')
p_topk = F.softmax(topk_logits, dim=-1)

# Top-p (p=0.85)
sorted_l, s_idx = torch.sort(raw_logits, descending=True)
cum_p = torch.cumsum(F.softmax(sorted_l, dim=-1), dim=-1)
mask_rm = cum_p > 0.85
mask_rm[1:] = mask_rm[:-1].clone()
mask_rm[0] = False
to_rm = mask_rm.scatter(0, s_idx, mask_rm)
topp_logits = raw_logits.clone()
topp_logits[to_rm] = float('-inf')
p_topp = F.softmax(topp_logits, dim=-1)

print(f"Logits Awal: {dict(zip(cand_words, raw_logits.tolist()))}")
print(f"P (T=0.2, Tajam/Konservatif) : {[round(p, 4) for p in p_cold.tolist()]}")
print(f"P (T=1.0, Netral/Alami)      : {[round(p, 4) for p in p_norm.tolist()]}")
print(f"P (T=2.0, Rata/Kreatif)      : {[round(p, 4) for p in p_warm.tolist()]}")
print(f"P (Top-k = 2)                : {[round(p, 4) for p in p_topk.tolist()]}")
print(f"P (Top-p = 0.85)             : {[round(p, 4) for p in p_topp.tolist()]}")

Logits Awal: ['cerdas': 4.5, 'hebat': 3.8, 'cepat': 3.0, 'luas': 1.2, 'lambat': 0.1]
P (T=0.2, Tajam/Konservatif) : [0.9675, 0.0312, 0.0013, 0.0000, 0.0000]
P (T=1.0, Netral/Alami)      : [0.5513, 0.2748, 0.1232, 0.0371, 0.0136]
P (T=2.0, Rata/Kreatif)      : [0.3664, 0.2575, 0.1741, 0.1118, 0.0902]
P (Top-k = 2)                : [0.6672, 0.3328, 0.0000, 0.0000, 0.0000]
P (Top-p = 0.85)             : [0.6672, 0.3328, 0.0000, 0.0000, 0.0000]


### 4. Parameter-Efficient Fine-Tuning: Modul LoRA PyTorch
Membuktikan bahwa LoRA $W_0 + \frac{\alpha}{r} (B \cdot A)$ menghemat parameter secara masif tanpa memodifikasi bobot dasar.

In [5]:
d_dim = 128
rank = 4
alpha = 8.0

W_0 = nn.Linear(d_dim, d_dim, bias=False)
W_0.weight.requires_grad = False  # DIBEKUKAN

lora_A = nn.Parameter(torch.randn(rank, d_dim) * 0.1)
lora_B = nn.Parameter(torch.zeros(d_dim, rank))  # Awalnya 0

x_sample = torch.randn(2, 4, d_dim)
h = W_0(x_sample) + ((x_sample @ lora_A.t()) @ lora_B.t()) * (alpha / rank)
h.sum().backward()

base_params = W_0.weight.numel()
lora_params = lora_A.numel() + lora_B.numel()
saving = (1.0 - (lora_params / base_params)) * 100.0

print('=== LOW-RANK ADAPTATION (LoRA) AUDIT ===')
print(f'Dimensi Bobot Model Dasar (W_0) : {W_0.weight.shape} ({base_params:,} bobot beku)')
print(f'Dimensi Adaptor LoRA A          : {lora_A.shape}')
print(f'Dimensi Adaptor LoRA B          : {lora_B.shape}')
print(f'Total Parameter Adaptor Terlatih: {lora_params:,} bobot')
print(f'Persentase Penghematan Parameter: {saving:.2f}%!')
print(f'Gradien W_0: {W_0.weight.grad is not None} (Beku) | Gradien LoRA A: {lora_A.grad is not None} (Aktif) | Gradien LoRA B: {lora_B.grad is not None} (Aktif)')

=== LOW-RANK ADAPTATION (LoRA) AUDIT ===
Dimensi Bobot Model Dasar (W_0) : torch.Size([128, 128]) (16,384 bobot beku)
Dimensi Adaptor LoRA A          : torch.Size([4, 128])
Dimensi Adaptor LoRA B          : torch.Size([128, 4])
Total Parameter Adaptor Terlatih: 1,024 bobot
Persentase Penghematan Parameter: 93.75%!
Gradien W_0: False (Beku) | Gradien LoRA A: True (Aktif) | Gradien LoRA B: True (Aktif)


### 5. Grounding & Audit Halusinasi Faktual
Membandingkan klaim generasi terhadap dokumen rujukan untuk menghitung Groundedness Score.

In [6]:
context_ref = 'ITERA berlokasi di Lampung Selatan dan memiliki 20.000 mahasiswa aktif pada tahun 2024.'
generated_claims = [
    'ITERA berlokasi di Lampung Selatan.',
    'ITERA memiliki 20.000 mahasiswa pada 2024.',
    'ITERA didirikan pada tahun 1945 oleh PBB.'
]

supported = [True, True, False]
score_grounded = sum(supported) / len(supported)
score_hallucination = 1.0 - score_grounded

print('=== AUDIT HALUSINASI FAKTUAL ===')
print(f"Konteks Rujukan: '{context_ref}'")
for i, (c, s) in enumerate(zip(generated_claims, supported), 1):
    verdict = 'TERVERIFIKASI' if s else 'HALUSINASI EKSTRINSIK'
    print(f"Klaim {i}: '{c}' -> {verdict}")
print(f'Skor Groundedness = {score_grounded:.2%} | Tingkat Halusinasi = {score_hallucination:.2%}')

=== AUDIT HALUSINASI FAKTUAL ===
Konteks Rujukan: 'ITERA berlokasi di Lampung Selatan dan memiliki 20.000 mahasiswa aktif pada tahun 2024.'
Klaim 1: 'ITERA berlokasi di Lampung Selatan.' -> TERVERIFIKASI
Klaim 2: 'ITERA memiliki 20.000 mahasiswa pada 2024.' -> TERVERIFIKASI
Klaim 3: 'ITERA didirikan pada tahun 1945 oleh PBB.' -> HALUSINASI EKSTRINSIK
Skor Groundedness = 66.67% | Tingkat Halusinasi = 33.33%
